# Brain-Inspired Replay: component ablation on Split MNIST (class-incremental)

Reproduces brain-inspired replay (BI-R, van de Ven et al., 2020, *Nature Communications*) on class-incremental Split MNIST and removes its components one at a time to see which one matters most.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Rough run time on a T4: about 60–75 minutes for 7 configurations × 3 seeds. Set `SEEDS = [1, 2]` in cell 3 for a faster run.

In [ ]:
# 1) Get the code and the extra packages (PyTorch is already installed on Colab)
!git clone -q https://github.com/GMvandeVen/brain-inspired-replay.git
%cd brain-inspired-replay
!pip install -q visdom tqdm
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# 2) Configurations
# On MNIST there are no conv layers, so BI-R = replay-through-feedback + conditional replay (GMM prior per class)
# + context gates in the decoder + distillation. Each 'BIR-no...' run removes exactly one of these.
CONFIGS = {
    "None (lower bound)":       "",
    "Standard GR":              "--replay=generative",
    "BI-R (full)":              "--replay=generative --brain-inspired",
    "BI-R − distillation":      "--replay=generative --feedback --prior=GMM --per-class --dg-gates",
    "BI-R − gating":            "--replay=generative --feedback --prior=GMM --per-class --distill",
    "BI-R − conditional replay":"--replay=generative --feedback --dg-gates --distill",
    "BI-R − feedback":          "--replay=generative --prior=GMM --per-class --dg-gates --distill",
}

In [ ]:
# 3) Run everything (prints one line per run)
import subprocess, re, time, json, os
SEEDS = [1, 2, 3]
results = {}   # name -> list of accuracies
os.makedirs("logs", exist_ok=True)
for name, flags in CONFIGS.items():
    results[name] = []
    for seed in SEEDS:
        t0 = time.time()
        cmd = f"python main_cl.py --experiment=splitMNIST --scenario=class {flags} --seed={seed} --no-save"
        out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        log = out.stdout + out.stderr
        tag = re.sub(r'[^A-Za-z0-9]+', '_', name)
        open(f"logs/{tag}_s{seed}.log", "w").write(log)
        m = re.search(r"Average accuracy over all 10 classes: ([0-9.]+)", log)
        acc = float(m.group(1)) if m else float("nan")
        results[name].append(acc)
        print(f"{name:28s} seed {seed}: {acc*100:6.2f}%   ({(time.time()-t0)/60:.1f} min)")
json.dump(results, open("ablation_results.json", "w"), indent=2)

In [ ]:
# 4) Table: mean ± std over seeds
import numpy as np, pandas as pd
rows = [(n, np.mean(a)*100, np.std(a)*100, len(a)) for n, a in results.items()]
df = pd.DataFrame(rows, columns=["Method", "Mean acc (%)", "Std (%)", "Seeds"]).round(2)
df.to_csv("ablation_results.csv", index=False)
df

In [ ]:
# 5) Plot
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(df["Method"], df["Mean acc (%)"], xerr=df["Std (%)"], color="#4C72B0")
ax.invert_yaxis()
ax.set_xlabel("Average test accuracy after all 5 tasks (%)")
ax.set_title("Split MNIST, class-incremental: BI-R ablation")
ax.set_xlim(0, 100)
for i, v in enumerate(df["Mean acc (%)"]):
    ax.text(v + 1, i, f"{v:.1f}", va="center")
plt.tight_layout(); plt.savefig("ablation_plot.png", dpi=150); plt.show()

In [ ]:
# 6) Download the outputs (table, plot, logs) for your GitHub repo
!zip -q -r ablation_outputs.zip ablation_results.csv ablation_results.json ablation_plot.png logs
from google.colab import files
files.download("ablation_outputs.zip")